# Applicator

**This is for the 2-way task**

I want to apply a theory to a response. In principle, the response should just be a sentence, but it needs to have been read and SpaCy'd first. So for the moment, I'll just assume that we're applying to sentences from the data file. Which, fortunately, is all encoded in the theory object, having been extracted from `bea_2way_data_pickle`.

Want two outputs: one of the results, one of the applications.

The theory itself will be the output from `bea_2way_theory_generator.ipynb/induce_theory`.

Want `apply_theory(theory, original_response_id)`.

Actually, a better plan might be to just have a function which applies the theory to the embedded trial set.

`apply_trial_set(theory)`

In [1]:
THEORY_ROOT_DIR='/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318'

In [2]:
import amati_bea as amati

In [3]:
import pandas as pd

from sklearn.metrics import f1_score, cohen_kappa_score

In [4]:
from operator import itemgetter
import pickle
import datetime
import re
import glob
import json
import os

## Get the data

Start off by getting the data we're applying the theory to:

In [5]:
JSON_DATA_FILE = "/Users/alistair.willis/repos/bea-2026/BEA Shared Task 2026 on Rubric-based Short Answer Scoring for German/updated/2way/ALICE_LP_train_2way__v2.json"

with open(JSON_DATA_FILE) as fIn:
    bea_data = json.load(fIn)

bea_data_with_amati = []

In [6]:
bea_data

[{'id': '73b0b485-5e89-40d8-ae68-4a650d4670c4',
  'question_id': '920df344-fe5c-4f48-ae64-a8458e8db40a',
  'question': 'Beurteile das Zerteilen der Äpfel hinsichlich der Geschwindigkeit des Verderblichkeitsprozesses (Hinweis: wenn Du fachliche Informationen zu chemischen Abläufen des Verderbens einbeziehen möchtest, kannst Du diese unter Angabe der Quellen recherchieren und einbinden).',
  'answer': 'Bei einem höheren Zerteilungsgrad ( hier das Zerteilen der Äpfel ) kommt es zu einer schnelleres Reaktionsgeschwindigkeit .\nDie hat man anhand unseres Modellversuches gesehen .\nAlso können wir feststellen , dass ein Stoff mit einem höheren Zerteilungsgrad einen Einfluss auf die Reaktionsgeschwindigkeit .\nSomit reagieren auch die zerschnittenden Äpfel schnelller .\n',
  'rubric': {'Incorrect': ['Die Schüler:innen beurteilen die Verderblichkeit der Äpfel nicht auf Basis des Kollisionsmodells',
    'Die Schüler:innen beurteilen die Verderblichkeit der Äpfel teilweise auf Basis des Kollisio

The test run in this case will apply the theory to a set of responses in json format, then create a new json file with the applied rule information in place.

We'll just be able to pull out the relevant data from the saved theory.

In [7]:
with open(
    "/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318/theory_0f235f9a-64f8-4790-917d-23e0a15077c5_2026-03-18_15:10:43.190962.pickle",
    "rb",
) as fIn:
    theory = pickle.load(fIn)

In [8]:
theory.keys()

dict_keys(['theory', 'original_question_id', 'question_id', 'all_questions_df', 'all_responses_df', 'all_text_df', 'rules'])

Let's find a trial response. First, need to pull out the question for that theory:

In [9]:
theory["question_id"]

'q_36'

And then get the associated responses:

In [10]:
theory["all_responses_df"].query("`question_id`==@theory['question_id']").query(
    "`partition`=='trial'"
).head()

,original_response_id,response_id,question_id,response,score,partition
3682,bff5b4ff-1376-4df0-820a-e2b8ad294555,r_1939,q_36,Die Teilchen haben die Eigenschaft bei kältere...,incorrect,trial
3683,c76a5fe9-d18d-40d6-917a-367c57aca94a,r_2043,q_36,Durch die Kühlung bewegen sich die Teilchen la...,incorrect,trial
3684,84535d1e-0014-49f8-9dfa-fc6b445d1fe3,r_2105,q_36,Niedrigere Temperatur- &gt; Niedrigerere Teilc...,incorrect,trial
3685,ef1fb3a9-00ae-445b-8e32-83e342601b99,r_4243,q_36,Allgemeine chemische Prozesse werden durch ein...,correct,trial
3686,3d0317b9-904b-4718-9ec5-88406a9951c8,r_4852,q_36,"Kühlung ist ein Vorgang , bei dem einem System...",incorrect,trial


OK, so now we should be able to pull out the cases we want to apply the theory to.

In [11]:
responses_json_ls = [
    x for x in bea_data if x["question_id"] == theory["original_question_id"]
]

responses_json_ls

[{'id': '639771c5-241b-4c35-921b-087896099ce8',
  'question_id': '0f235f9a-64f8-4790-917d-23e0a15077c5',
  'question': 'Erläutere die konservierende Wirkung einer Kühlung aus reaktionskinetischer Perspektive und beziehe Dich auch hierbei auf die unterschiedlichen Betrachtungsebenen der Chemie.',
  'answer': 'Die Gärung oder das verfaulen ist auch eine Art Reaktion .\nDurch die niedrigen Temperaturen gibt es im verwahrten Produkt weniger Reaktionen , da die Geschwindigkeit der Teilchen viel niedriger ist .\nAußerdem sind kalte Räume für viele Bakterien und Schädlinge kein guter Lebensraum\n',
  'rubric': {'Incorrect': ['Die Schüler:innen erklären den konservierenden Effekt der Kühlung nicht aus reaktionskinetischer Perspektive.',
    'Die Scüler:innen erklären den konservierenden Effekt der Kühlung aus reaktionskinetischer Perspektive, wenden kinetisches Wissen aber nur teilweise (wenigstens ein Aspekt) auf das konkrete Phänomen an.'],
   'Correct': 'Die Schüler:innen eklären den konser

Now we need to get the data into the right form to apply the theory.

I'm going to apply the theory to all the possible cases (ie. training and trial data).

And we can get the dataframe version of the data from the theory structure:

In [12]:
application_question_ss = (
    theory["all_questions_df"]
    .set_index("original_question_id")
    .T[theory["original_question_id"]]
)
application_question_ss

question_id                                                    q_36
question          Erläutere die konservierende Wirkung einer Küh...
correct           Die Schüler:innen eklären den konservierenden ...
incorrect_1       Die Schüler:innen erklären den konservierenden...
incorrect_2       Die Scüler:innen erklären den konservierenden ...
correct_id                                                     c_36
incorrect_1_id                                                 i_36
incorrect_2_id                                                 j_36
Name: 0f235f9a-64f8-4790-917d-23e0a15077c5, dtype: object

In [13]:
application_responses_df = theory["all_responses_df"][
    theory["all_responses_df"]["question_id"] == application_question_ss["question_id"]
]

application_responses_df

,original_response_id,response_id,question_id,response,score,partition
3574,d49e5db1-8757-4718-8b51-7ec94f328fc5,r_0013,q_36,Durch die Kälte finden Biochemische Reaktionen...,incorrect,train
3575,98fa8dc4-1c7e-4faa-b618-f553ce0b8a49,r_0078,q_36,Durch das kühlen der Apfel ist die Reaktion ( ...,correct,train
3576,a4d97e5f-b2cf-484f-855b-ef248aa5e287,r_0110,q_36,"Durch die Lagerung im kühleren Medium , kann d...",incorrect,train
3577,80539cd0-06d8-4450-a788-3448ab64d2dc,r_0208,q_36,Die Kühlung verlangsamt die Reaktionen .\n,incorrect,train
3578,77b3a36c-dc5e-466a-b85b-44bba5abbb8f,r_0311,q_36,"Dadurch , dass man Lebensmittel kühlt , bleibt...",incorrect,train
...,...,...,...,...,...,...
3690,e9628083-3a17-4725-96b1-42f2728945fd,r_6075,q_36,Da die Temperatur im Kühlschrank geringer ist ...,correct,trial
3691,0dd6baad-11e7-47a1-ac9e-18249682cf7d,r_6897,q_36,"Die Reaktion der Teilchen läuft langsamer ab ,...",correct,trial
3692,ff3417b1-32e5-4daa-8be8-b573be533ba2,r_7405,q_36,Die Temperaturabnahme verringert die Reaktions...,incorrect,trial
3693,d5c7db2f-b5a8-4b99-80ad-abc0c5d44b6b,r_7679,q_36,Durch die Kühlung bewegen sich die Teilchen ni...,incorrect,trial


In [14]:
application_texts_used = [
    application_question_ss["question_id"],
    application_question_ss["correct_id"],
    application_question_ss["incorrect_1_id"],
    application_question_ss["incorrect_2_id"],
] + list(application_responses_df["response_id"])

application_text_df = theory["all_text_df"].query("`id`.isin(@application_texts_used)")[
    ["id", "i", "lemma"]
]

application_text_df

,id,i,lemma
871,q_36,0,erläuter
872,q_36,1,der
873,q_36,2,konservierend
874,q_36,3,wirkung
875,q_36,4,ein
...,...,...,...
128990,r_7740,31,niedrig
128991,r_7740,32,gehaltend
128992,r_7740,33,temperatur
128993,r_7740,34,verlangsamen


And now we should be able to apply the learner to this data. Let's see what happens...

In [15]:
evaluation_results_df = amati.evaluate_theory(
    theory["theory"],
    responses_df=application_responses_df,
    text_df=application_text_df,
    rulesstring=theory["rules"],
    question_ss=application_question_ss,
)

In [16]:
application_results_dict = amati.apply_theory(
    theory["theory"],
    responses_df=application_responses_df,
    text_df=application_text_df,
    rulesstring=theory["rules"],
    question_ss=application_question_ss,
)

In [17]:
evaluation_results_df

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
response_id,,,,,,,,,,,,,,,,,,,,
r_0013,NaN,NaN,incorrect,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_0078,NaN,correct,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_0110,incorrect,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_0208,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_0311,incorrect,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
r_6075,NaN,correct,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_6897,NaN,correct,incorrect,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,incorrect,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r_7405,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


OK, now to get the rules in the right form, we want a function for converting a rule description into a Python dict:

In [18]:
def get_json_from_rule(rule):
    if rule["selected_rule"][0] == "rule1":
        return {
            "amati": {
                "literals": [
                    {
                        "pred": "contains",
                        "document": "student_response",
                        "lemma": [x[1] for x in rule["parameters"] if x[0] == 1][0],
                        "id": "@1",
                    }
                ],
                "prediction": rule["predicted_grade"][0],
            }
        }
    elif rule["selected_rule"][0] == "rule2":
        return {
            "amati": {
                "literals": [
                    {
                        "pred": "contains",
                        "document": "student_response",
                        "lemma": [x[1] for x in rule["parameters"] if x[0] == 1][0],
                        "id": "@1",
                    },
                    {
                        "pred": "contains",
                        "document": "student_response",
                        "lemma": [x[1] for x in rule["parameters"] if x[0] == 2][0],
                        "id": "@2",
                    },
                    {"pred": "precedes", "order": ["@1", "@2"]},
                ],
                "prediction": rule["predicted_grade"][0],
            }
        }
    elif rule["selected_rule"][0] == "rule3":
        return {
            "amati": {
                "literals": [
                    {
                        "pred": "contains",
                        "document": "student_response",
                        "lemma": [x[1] for x in rule["parameters"] if x[0] == 1][0],
                        "id": "@1",
                    },
                    {
                        "pred": "contains",
                        "document": "student_response",
                        "lemma": [x[1] for x in rule["parameters"] if x[0] == 2][0],
                        "id": "@2",
                    },
                ],
                "prediction": rule["predicted_grade"][0],
            }
        }
    
    
    # elif rule["selected_rule"][0] == "ki_rule1":
    #     return {
    #         "amati": {
    #             "literals": [
    #                 {
    #                     "pred": "contains",
    #                     "document": "student_response",
    #                     "lemma": [x[1] for x in rule["parameters"] if x[0] == 1][0],
    #                     "id": "@1",
    #                 },
    #                 {
    #                     "pred": "contains",
    #                     "document": "reference_answer",
    #                     "lemma": [x[1] for x in rule["parameters"] if x[0] == 1][0],
    #                     "id": "@2",
    #                 },
    #                 {
    #                     "pred": "contains",
    #                     "negated": True,
    #                     "document": "question",
    #                     "lemma": [x[1] for x in rule["parameters"] if x[0] == 1][0],
    #                     "id": "@3",
    #                 },
    #             ],
    #             "prediction": rule["predicted_grade"][0],
    #         }
    #     }
        
    elif rule["selected_rule"][0] == "ki_rule1":
        return {
            "amati": {
                "literals": [],
                "prediction": rule["predicted_grade"][0],
            }
        }




    else:
        assert False

In [19]:
#get_json_from_rule(application_results_dict["r_0320"][0])
application_results_dict["r_6897"][0]

{'result': 'SATISFIABLE',
 'selected_rule': ('rule1', 'selected_rule(rule1)'),
 'predicted_grade': ('correct', 'predicted_grade(correct)'),
 'parameters': [(1, '"zusammenstoß"', 'parameter(1,"zusammenstoß")'),
  (2, '"teilche"', 'parameter(2,"teilche")')],
 'positive_covered': ['r_0078',
  'r_0473',
  'r_0650',
  'r_0805',
  'r_1929',
  'r_2106',
  'r_2236',
  'r_2536',
  'r_2568',
  'r_2769',
  'r_2825',
  'r_3147',
  'r_3957',
  'r_5553',
  'r_5984',
  'r_6092',
  'r_6421',
  'r_6699',
  'r_6753',
  'r_7117'],
 'negative_covered': ['r_3793', 'r_6492', 'r_6593'],
 'evaluations': {'precision': 86,
  'compression': 17,
  'coverage': 17,
  'accuracy': 86},
 'positive_df':     response_id question_id      score  covered
 0        r_0013        q_36  incorrect    False
 1        r_0078        q_36    correct     True
 3        r_0208        q_36  incorrect    False
 5        r_0321        q_36  incorrect    False
 6        r_0350        q_36  incorrect    False
 ..          ...         ...

So for the complete set, we need to decide whether we're using the first applied rule or the final applied rule (we're using the first, as it's more intuitive to go with the first application), and what the default is. In this case, the default will be incorrect, and the `amati` feature will have  an empty set as the value.

So let's now run through the json dictionary and add the `amati` feature to each response in the file:

In [20]:
bea_data

[{'id': '73b0b485-5e89-40d8-ae68-4a650d4670c4',
  'question_id': '920df344-fe5c-4f48-ae64-a8458e8db40a',
  'question': 'Beurteile das Zerteilen der Äpfel hinsichlich der Geschwindigkeit des Verderblichkeitsprozesses (Hinweis: wenn Du fachliche Informationen zu chemischen Abläufen des Verderbens einbeziehen möchtest, kannst Du diese unter Angabe der Quellen recherchieren und einbinden).',
  'answer': 'Bei einem höheren Zerteilungsgrad ( hier das Zerteilen der Äpfel ) kommt es zu einer schnelleres Reaktionsgeschwindigkeit .\nDie hat man anhand unseres Modellversuches gesehen .\nAlso können wir feststellen , dass ein Stoff mit einem höheren Zerteilungsgrad einen Einfluss auf die Reaktionsgeschwindigkeit .\nSomit reagieren auch die zerschnittenden Äpfel schnelller .\n',
  'rubric': {'Incorrect': ['Die Schüler:innen beurteilen die Verderblichkeit der Äpfel nicht auf Basis des Kollisionsmodells',
    'Die Schüler:innen beurteilen die Verderblichkeit der Äpfel teilweise auf Basis des Kollisio

In [21]:
DEFAULT_GRADE = "incorrect"

for response_id, applications in application_results_dict.items():

    original_response_id = (
        theory["all_responses_df"]
        .set_index("response_id")
        .loc[response_id, "original_response_id"]
    )

    if not applications:
        amati_dict = {"amati": {'literals':[], "prediction": DEFAULT_GRADE}}
    else:
        amati_dict = get_json_from_rule(application_results_dict[response_id][0])

    for c in bea_data:
        if c["id"] == original_response_id:
            c.update(amati_dict)
            bea_data_with_amati.append(c)

In [22]:
bea_data_with_amati

[{'id': 'd49e5db1-8757-4718-8b51-7ec94f328fc5',
  'question_id': '0f235f9a-64f8-4790-917d-23e0a15077c5',
  'question': 'Erläutere die konservierende Wirkung einer Kühlung aus reaktionskinetischer Perspektive und beziehe Dich auch hierbei auf die unterschiedlichen Betrachtungsebenen der Chemie.',
  'answer': 'Durch die Kälte finden Biochemische Reaktionen weniger bis garnicht statt .\n',
  'rubric': {'Incorrect': ['Die Schüler:innen erklären den konservierenden Effekt der Kühlung nicht aus reaktionskinetischer Perspektive.',
    'Die Scüler:innen erklären den konservierenden Effekt der Kühlung aus reaktionskinetischer Perspektive, wenden kinetisches Wissen aber nur teilweise (wenigstens ein Aspekt) auf das konkrete Phänomen an.'],
   'Correct': 'Die Schüler:innen eklären den konservierenden Effekt der Kühlung aus reaktionskintischer Perspektve un wenden kinetisches Wissen umfassend auf das konkrete Phänomen an.'},
  'score': 'Incorrect',
  'amati': {'literals': [{'pred': 'contains',
   

OK, that seems to be working OK (fingers crossed).

So now let's apply to the whole set of theories. I'll need to do this twice: once for the training data and once for the trial data.

## Training data

In [23]:
JSON_DATA_FILE = "/Users/alistair.willis/repos/bea-2026/BEA Shared Task 2026 on Rubric-based Short Answer Scoring for German/updated/2way/ALICE_LP_train_2way__v2.json"

with open(JSON_DATA_FILE) as fIn:
    bea_data = json.load(fIn)

bea_data_with_amati = []

In [25]:
for filename in glob.glob(os.path.join(THEORY_ROOT_DIR, "*.pickle")):
    print(filename)

    with open(filename, "rb") as fIn:
        theory = pickle.load(fIn)

    application_question_ss = (
        theory["all_questions_df"]
        .set_index("original_question_id")
        .T[theory["original_question_id"]]
    )

    application_responses_df = theory["all_responses_df"][
        theory["all_responses_df"]["question_id"]
        == application_question_ss["question_id"]
    ]

    application_texts_used = [
        application_question_ss["question_id"],
        application_question_ss["correct_id"],
        application_question_ss["incorrect_1_id"],
        application_question_ss["incorrect_2_id"],
    ] + list(application_responses_df["response_id"])

    application_text_df = theory["all_text_df"].query(
        "`id`.isin(@application_texts_used)"
    )[["id", "i", "lemma"]]

   
    application_results_dict = amati.apply_theory(
    theory["theory"],
    responses_df=application_responses_df,
    text_df=application_text_df,
    rulesstring=theory["rules"],
    question_ss=application_question_ss,
)
    

    DEFAULT_GRADE = "incorrect"

    for response_id, applications in application_results_dict.items():

        original_response_id = (
        theory["all_responses_df"]
        .set_index("response_id")
        .loc[response_id, "original_response_id"]
    )

        if not applications:
            amati_dict = {"amati": {'literals':[], "prediction": DEFAULT_GRADE}}
        else:
            amati_dict = get_json_from_rule(application_results_dict[response_id][0])

        for c in bea_data:
            if c["id"] == original_response_id:
                c.update(amati_dict)
                bea_data_with_amati.append(c)





/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318/theory_5f7bd982-606b-45e4-bf39-89db6714a621_2026-03-18_12:52:28.028319.pickle
/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318/theory_35d897f0-c1b4-422c-9a33-43e7da1ffe27_2026-03-18_12:57:35.168998.pickle
/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318/theory_52f6e329-eea6-423b-8d6a-047448d02c3f_2026-03-18_15:45:58.021429.pickle
/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318/theory_c405c4a8-83f2-4ddf-b0cf-941a3356b251_2026-03-18_16:59:35.923009.pickle
/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318/theory_0235896f-1570-418a-a34d-7bc3ea0f82b9_2026-03-18_14:07:56.575271.pickle
/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318/theory_920df344-fe5c-4f48-ae64-a8458e8db40a_2026-03-18_15:24:00.591946.pickle
/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260318/theory_c51b89a6-0219-4eee-bb89-b2b5168998f5_2026-03-18_14:12:46.706850

KeyError: 'Witnesses'

In [ ]:
with open('train_2way_with_prediction.json', 'w') as fOut:
    json.dump(bea_data_with_amati, fOut)

## Trial data

In [ ]:
JSON_DATA_FILE = "/Users/alistair.willis/repos/bea-2026/BEA Shared Task 2026 on Rubric-based Short Answer Scoring for German/updated/2way/ALICE_LP_trial_2way___v2.json"

with open(JSON_DATA_FILE) as fIn:
    bea_data = json.load(fIn)

bea_data_with_amati = []

In [ ]:
for filename in glob.glob(
    "/Users/alistair.willis/repos/bea-2026/2-way/theories/run_20260316/*.pickle"
):

    print(filename)

    with open(filename, "rb") as fIn:
        theory = pickle.load(fIn)

    application_question_ss = (
        theory["all_questions_df"]
        .set_index("original_question_id")
        .T[theory["original_question_id"]]
    )

    application_responses_df = theory["all_responses_df"][
        theory["all_responses_df"]["question_id"]
        == application_question_ss["question_id"]
    ]

    application_texts_used = [
        application_question_ss["question_id"],
        application_question_ss["correct_id"],
        application_question_ss["incorrect_1_id"],
        application_question_ss["incorrect_2_id"],
    ] + list(application_responses_df["response_id"])

    application_text_df = theory["all_text_df"].query(
        "`id`.isin(@application_texts_used)"
    )[["id", "i", "lemma"]]

    application_results_dict = amati.apply_theory(
        theory["theory"],
        responses_df=application_responses_df,
        text_df=application_text_df,
        rulesstring=theory["rules"],
        question_ss=application_question_ss,
    )

    DEFAULT_GRADE = "incorrect"

    for response_id, applications in application_results_dict.items():

        original_response_id = (
            theory["all_responses_df"]
            .set_index("response_id")
            .loc[response_id, "original_response_id"]
        )

        if not applications:
            amati_dict = {"amati": {"literals": [], "prediction": DEFAULT_GRADE}}
        else:
            amati_dict = get_json_from_rule(application_results_dict[response_id][0])

        for c in bea_data:
            if c["id"] == original_response_id:
                c.update(amati_dict)
                bea_data_with_amati.append(c)

In [ ]:
with open('trial_2way_with_prediction.json', 'w') as fOut:
    json.dump(bea_data_with_amati, fOut)